In [1]:
!pip install -q pandas gdown librosa regex tqdm
!apt-get install -y ffmpeg


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.


In [2]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [3]:
import os

BASE_DIR = "/content/drive/MyDrive"

INPUT_CSV = os.path.join(BASE_DIR, "input_cs_data.csv")
WEBM_DIR  = os.path.join(BASE_DIR, "Audio data set")
WAV_DIR   = os.path.join(BASE_DIR, "Audio Wav data zero shot set")
OUTPUT_CSV = os.path.join(BASE_DIR, "output_clean_cs_data.csv")
# WAV_FOLDER_ID = "1oiks5g0UPuZsReUkpXCydny24vLFuO7p"


os.makedirs(WAV_DIR, exist_ok=True)


In [ ]:
import re

def remove_tamil_suffix(text):
    """
    Removes Tamil suffixes attached to English words using '-'
    Example: meeting-க்கு -> meeting
    """
    # Match: EnglishWord-தமிழ்
    pattern = r'([A-Za-z]+)-[^\s]+'
    return re.sub(pattern, r'\1', text)


In [4]:
import subprocess
import librosa

def convert_webm_to_wav(webm_path, wav_path):
    subprocess.run(
        ["ffmpeg", "-y", "-i", webm_path, wav_path],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

def get_audio_duration(wav_path):
    y, sr = librosa.load(wav_path, sr=None)
    return round(librosa.get_duration(y=y, sr=sr), 2)


In [5]:
# not need to run this command
!pip install -U pydrive2


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
              

In [6]:
#not need to run this block
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)


KeyboardInterrupt: 

In [7]:
def get_drive_file_link(filename, folder_id):
    query = (
        f"'{folder_id}' in parents and "
        f"title='{filename}' and trashed=false"
    )

    file_list = drive.ListFile({'q': query}).GetList()

    if not file_list:
        return ""

    file_id = file_list[0]['id']
    return f"https://drive.google.com/file/d/{file_id}/view"


In [8]:
import pandas as pd

MAX_TOTAL_SECONDS = 5 * 60 * 60  # 5 hours
total_duration = 0.0

df = pd.read_csv(INPUT_CSV)
df.columns = df.columns.str.strip()  # safety

output_rows = []

for idx, row in df.iterrows():
    if total_duration >= MAX_TOTAL_SECONDS:
        print("⏹ Reached 5 hours limit. Stopping processing.")
        break

    script_text = row["script_text"]
    filename = row["file_name"]

    # Clean text
    # cleaned_text = remove_tamil_suffix(script_text)

    # File paths
    webm_file = filename if filename.endswith(".webm") else filename + ".webm"
    webm_path = os.path.join(WEBM_DIR, webm_file)

    wav_file = os.path.splitext(webm_file)[0] + ".wav"
    wav_path = os.path.join(WAV_DIR, wav_file)



    if not os.path.exists(webm_path):
        print(f"⚠️ Missing file: {webm_path}")
        continue


    convert_webm_to_wav(webm_path, wav_path)

    # Duration
    duration = get_audio_duration(wav_path)

    # Check again before adding
    if total_duration + duration > MAX_TOTAL_SECONDS:
        print("⏹ Next file exceeds 5 hours limit. Skipping.")
        break

    total_duration += duration
    # wav_http_link = get_drive_file_link(wav_file, WAV_FOLDER_ID)

    output_rows.append({
        "script_text": script_text,
        "audio_wav_path": wav_path,
        "duration_seconds": duration
    })

    if idx % 10 == 0:
        print(f"⏳ Processed {idx} files | Total duration: {total_duration/3600:.2f} hrs")

# Save output CSV
out_df = pd.DataFrame(output_rows)
out_df.to_csv(OUTPUT_CSV, index=False)

print("✅ Processing complete!")
print(f"🎧 Total audio duration: {total_duration/3600:.2f} hours")
print("📄 Output CSV saved at:", OUTPUT_CSV)


⏳ Processed 0 files | Total duration: 0.00 hrs
⏳ Processed 10 files | Total duration: 0.02 hrs
⏳ Processed 20 files | Total duration: 0.03 hrs
⏳ Processed 30 files | Total duration: 0.04 hrs
⏳ Processed 40 files | Total duration: 0.06 hrs
⏳ Processed 50 files | Total duration: 0.07 hrs
⏳ Processed 60 files | Total duration: 0.08 hrs
⏳ Processed 70 files | Total duration: 0.10 hrs
⏳ Processed 80 files | Total duration: 0.11 hrs
⏳ Processed 90 files | Total duration: 0.13 hrs
⏳ Processed 100 files | Total duration: 0.14 hrs
⏳ Processed 110 files | Total duration: 0.15 hrs
⏳ Processed 120 files | Total duration: 0.16 hrs
⏳ Processed 130 files | Total duration: 0.18 hrs
⏳ Processed 140 files | Total duration: 0.19 hrs
⏳ Processed 150 files | Total duration: 0.20 hrs
⏳ Processed 160 files | Total duration: 0.22 hrs
⏳ Processed 170 files | Total duration: 0.23 hrs
⏳ Processed 180 files | Total duration: 0.24 hrs
⏳ Processed 190 files | Total duration: 0.26 hrs
⏳ Processed 200 files | Total d